In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/enhancer-detection/weak_100.txt
/kaggle/input/enhancer-detection/strong_742.txt
/kaggle/input/enhancer-detection/strong_100.txt
/kaggle/input/enhancer-detection/enhancer.ind.txt
/kaggle/input/enhancer-detection/enhancer.cv.txt
/kaggle/input/enhancer-detection/non.cv.txt
/kaggle/input/enhancer-detection/non.ind.txt
/kaggle/input/enhancer-detection/weak_742.txt


LONGFORMER


In [2]:
import torch
import torch.nn as nn
import numpy as np
import time
import matplotlib.pyplot as plt
from transformers import LongformerTokenizer, LongformerModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, matthews_corrcoef, recall_score, roc_curve, auc
from collections import Counter
import random, re
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
from torch.optim.lr_scheduler import CosineAnnealingLR  # Import the missing CosineAnnealingLR

# --- Reproducibility ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# --- Utilities ---
def clean_sequence(seq):
    return re.sub('[^ATCG]', '', seq.upper())

def load_fasta_clean(path):
    with open(path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    return seq.translate(str.maketrans("ATCG", "TAGC"))[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

def augment_sequence(seq):
    bases = ['A','T','C','G']
    seq_list = list(seq)
    for i in range(len(seq)):
        if random.random() < 0.05:
            seq_list[i] = random.choice([b for b in bases if b != seq_list[i]])
    kmers = ["".join(seq_list[i:i+6]) for i in range(len(seq_list)-5)]
    random.shuffle(kmers)
    return " ".join(kmers)

# --- Dataset ---
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=128, augment=False):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.augment = augment

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        if self.augment and random.random() < 0.5:
            seq = augment_sequence(seq)
        else:
            seq = kmer_tokenize(seq)
        encoding = self.tokenizer(seq, padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float)
        }

# --- Longformer Model Class ---
class LongformerOnly(nn.Module):
    def __init__(self, longformer_name="allenai/longformer-base-4096", dropout=0.3):
        super().__init__()
        self.longformer = LongformerModel.from_pretrained(longformer_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.longformer.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.longformer(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output  # [batch_size, hidden_dim]
        logits = self.classifier(self.dropout(pooled_output)).squeeze(-1)
        return logits

# --- Evaluation Utility ---
def evaluate(model, loader, device):
    model.eval()
    preds, trues = [], []
    tp, tn, fp, fn = 0, 0, 0, 0  # Initialize confusion matrix counts
    all_probs = []  # List to store all probability values

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)
            logits = model(input_ids, attention_mask).squeeze()
            probs = torch.sigmoid(logits)
            preds.extend((probs > 0.5).int().cpu().tolist())
            trues.extend(label.int().cpu().tolist())
            all_probs.extend(probs.cpu().tolist())  # Collect all probs

            # Update confusion matrix
            for true, pred in zip(label.int().cpu().tolist(), (probs > 0.5).int().cpu().tolist()):
                if true == 1 and pred == 1:
                    tp += 1  # True Positive
                elif true == 0 and pred == 0:
                    tn += 1  # True Negative
                elif true == 0 and pred == 1:
                    fp += 1  # False Positive
                elif true == 1 and pred == 0:
                    fn += 1  # False Negative

    # Accuracy
    accuracy = accuracy_score(trues, preds)
    # F1 Score
    f1 = f1_score(trues, preds)
    # Specificity
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    # Sensitivity
    sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0
    # MCC
    mcc = matthews_corrcoef(trues, preds)

    # AUC - Ensure probs are in numpy format and length matches
    all_probs_np = np.array(all_probs).flatten()
    trues_np = np.array(trues)
    auc = roc_auc_score(trues_np, all_probs_np)

    print(f"Accuracy: {accuracy:.4f} | F1 Score: {f1:.4f} | Sensitivity (SN): {sensitivity:.4f} | Specificity (SP): {specificity:.4f} | MCC: {mcc:.4f} | AUC: {auc:.4f}")
    print("Confusion Matrix:\n", [[tp, fp], [fn, tn]])

    return accuracy, f1, sensitivity, specificity, mcc, auc

# === STAGE 1: ENHANCER VS NON-ENHANCER ===
def prepare_stage1():
    enhancer_train = load_fasta_clean("/kaggle/input/enhancer-detection/enhancer.cv.txt")
    non_enhancer_train = load_fasta_clean("/kaggle/input/enhancer-detection/non.cv.txt")
    
    enhancer_test = load_fasta_clean("/kaggle/input/enhancer-detection/enhancer.ind.txt")
    non_enhancer_test = load_fasta_clean("/kaggle/input/enhancer-detection/non.ind.txt")

    min_len_train = min(len(enhancer_train), len(non_enhancer_train))
    enhancer_train = enhancer_train[:min_len_train]
    non_enhancer_train = non_enhancer_train[:min_len_train]

    min_len_test = min(len(enhancer_test), len(non_enhancer_test))
    enhancer_test = enhancer_test[:min_len_test]
    non_enhancer_test = non_enhancer_test[:min_len_test]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for seq in enhancer_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(1)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(1)
    for seq in non_enhancer_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(0)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(0)

    for seq in enhancer_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(1)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(1)
    for seq in non_enhancer_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(0)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(0)

    return X_train, y_train, X_test, y_test


# === STAGE 2: STRONG VS WEAK ENHANCERS ===
def prepare_stage2():
    strong_train = load_fasta_clean("/kaggle/input/enhancer-detection/strong_742.txt")
    weak_train = load_fasta_clean("/kaggle/input/enhancer-detection/weak_742.txt")

    strong_test = load_fasta_clean("/kaggle/input/enhancer-detection/strong_100.txt")
    weak_test = load_fasta_clean("/kaggle/input/enhancer-detection/weak_100.txt")

    max_len_train = max(len(strong_train), len(weak_train))
    strong_train = (strong_train * ((max_len_train // len(strong_train)) + 1))[:max_len_train]
    weak_train = (weak_train * ((max_len_train // len(weak_train)) + 1))[:max_len_train]

    max_len_test = max(len(strong_test), len(weak_test))
    strong_test = (strong_test * ((max_len_test // len(strong_test)) + 1))[:max_len_test]
    weak_test = (weak_test * ((max_len_test // len(weak_test)) + 1))[:max_len_test]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for seq in strong_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(1)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(1)
    for seq in weak_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(0)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(0)

    for seq in strong_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(1)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(1)
    for seq in weak_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(0)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(0)

    return X_train, y_train, X_test, y_test

def train_stage1():
    tokenizer = LongformerTokenizer.from_pretrained("allenai/longformer-base-4096")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Prepare data
    X_train, y_train, X_test, y_test = prepare_stage1()
    print("Train Label Distribution:", Counter(y_train))
    print("Test Label Distribution:", Counter(y_test))

    train_ds = SimpleDataset(X_train, y_train, tokenizer)
    test_ds = SimpleDataset(X_test, y_test, tokenizer)
    batch_size = 8  # Starting batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    model = LongformerOnly(dropout=0.3).to(device)  # Using Longformer only
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # Learning rate scheduler (Cosine Annealing)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    best_f1 = 0
    for epoch in range(10):  # 10 epochs
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 1 - Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask).squeeze()
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")
        evaluate(model, test_loader, device)

        # Update learning rate based on scheduler
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                label = batch["label"].to(device)
                logits = model(input_ids, attention_mask).squeeze()
                preds.extend((torch.sigmoid(logits) > 0.5).int().cpu().tolist())
                trues.extend(label.int().cpu().tolist())
        f1 = f1_score(trues, preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "enhancer_stage1.pth")
            print("✅ Stage 1 model saved.")

def train_stage2():
    tokenizer = LongformerTokenizer.from_pretrained("allenai/longformer-base-4096")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Prepare data
    X_train, y_train, X_test, y_test = prepare_stage2()
    print("Train Label Distribution:", Counter(y_train))
    print("Test Label Distribution:", Counter(y_test))

    train_ds = SimpleDataset(X_train, y_train, tokenizer)
    test_ds = SimpleDataset(X_test, y_test, tokenizer)
    batch_size = 8  # Starting batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    model = LongformerOnly(dropout=0.3).to(device)  # Using Longformer only
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # Learning rate scheduler (Cosine Annealing)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    best_f1 = 0
    for epoch in range(10):  # 10 epochs
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 2 - Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask).squeeze()
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")
        evaluate(model, test_loader, device)

        # Update learning rate based on scheduler
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                label = batch["label"].to(device)
                logits = model(input_ids, attention_mask).squeeze()
                preds.extend((torch.sigmoid(logits) > 0.5).int().cpu().tolist())
                trues.extend(label.int().cpu().tolist())
        f1 = f1_score(trues, preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "enhancer_stage2.pth")
            print("✅ Stage 2 model saved.")

# === MAIN ===
if __name__ == "__main__":
    train_stage1()
    train_stage2()


Train Label Distribution: Counter({1: 2968, 0: 2968})
Test Label Distribution: Counter({1: 400, 0: 400})


pytorch_model.bin:   0%|          | 0.00/597M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/597M [00:00<?, ?B/s]


Stage 1 - Epoch 1:   0%|          | 0/742 [00:00<?, ?it/s]Input ids are automatically padded to be a multiple of `config.attention_window`: 512

Stage 1 - Epoch 1: 100%|██████████| 742/742 [21:17<00:00,  1.72s/it]


Epoch 1 Loss: 0.6705
Accuracy: 0.6312 | F1 Score: 0.6326 | Sensitivity (SN): 0.6350 | Specificity (SP): 0.6275 | MCC: 0.2625 | AUC: 0.6750
Confusion Matrix:
 [[254, 149], [146, 251]]
✅ Stage 1 model saved.


...

Epoch 8 Loss: 0.6942
Accuracy: 0.5000 | F1 Score: 0.0000 | Sensitivity (SN): 0.0000 | Specificity (SP): 1.0000 | MCC: 0.0000 | AUC: 0.4362
Confusion Matrix:
 [[0, 0], [500, 500]]


Stage 2 - Epoch 9: 100%|██████████| 928/928 [24:36<00:00,  1.59s/it]


Epoch 9 Loss: 0.6941
Accuracy: 0.5000 | F1 Score: 0.6667 | Sensitivity (SN): 1.0000 | Specificity (SP): 0.0000 | MCC: 0.0000 | AUC: 0.4364
Confusion Matrix:
 [[500, 500], [0, 0]]


Stage 2 - Epoch 10: 100%|██████████| 928/928 [24:35<00:00,  1.59s/it]


Epoch 10 Loss: 0.6937
Accuracy: 0.5000 | F1 Score: 0.6667 | Sensitivity (SN): 1.0000 | Specificity (SP): 0.0000 | MCC: 0.0000 | AUC: 0.4346
Confusion Matrix:
 [[500, 500], [0, 0]]


LONGFORMER + CNN


In [5]:
import torch
import torch.nn as nn
import numpy as np
from transformers import LongformerTokenizer, LongformerModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, matthews_corrcoef, roc_auc_score
from collections import Counter
import random
import re
from tqdm import tqdm

# --- Reproducibility ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# --- Data Utilities ---
def clean_sequence(seq):
    seq = seq.upper()
    return re.sub('[^ATCG]', '', seq)

def load_fasta_clean(file_path):
    with open(file_path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    complement = str.maketrans("ATCG", "TAGC")
    return seq.translate(complement)[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

# --- Shared Dataset Class ---
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=512):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        encoding = self.tokenizer(self.sequences[idx], padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.float),
        }

# --- Longformer + CNN Model ---
class LongformerCNN(nn.Module):
    def __init__(self, dropout_rate=0.3, kernel_size=1, num_filters=64):
        super().__init__()
        # Load pre-trained Longformer model
        self.longformer = LongformerModel.from_pretrained("allenai/longformer-base-4096")
        
        # CNN Layer with smaller kernel size
        self.conv1 = nn.Conv1d(in_channels=self.longformer.config.hidden_size, out_channels=num_filters, kernel_size=kernel_size)
        self.pool = nn.AdaptiveMaxPool1d(1)  # Global max pooling
        
        # Fully connected layer
        self.fc1 = nn.Linear(num_filters, 1)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout_rate)
        
    def forward(self, input_ids, attention_mask):
        # Pass through Longformer
        outputs = self.longformer(input_ids=input_ids, attention_mask=attention_mask)
        # Get the pooled output (CLS token representation)
        x = outputs.pooler_output
        
        # Reshape to fit CNN (batch_size, channels, sequence_length)
        x = x.unsqueeze(2)  # Add sequence length dimension (batch_size, 768, 1)
        
        # Apply convolution
        x = self.conv1(x)   # Apply convolution
        x = self.pool(x)    # Global max pooling
        x = x.squeeze(2)    # Squeeze sequence dimension
        
        # Pass through dropout and fully connected layer
        x = self.dropout(x)
        x = self.fc1(x)
        
        return x

# --- Evaluation Utility ---
def evaluate(model, loader, device):
    model.eval()
    preds, trues = [], []
    tp, tn, fp, fn = 0, 0, 0, 0  # Initialize confusion matrix counts
    all_probs = []  # List to store all probability values

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)
            logits = model(input_ids, attention_mask).squeeze()
            probs = torch.sigmoid(logits)
            preds.extend((probs > 0.5).int().cpu().tolist())
            trues.extend(label.int().cpu().tolist())
            all_probs.extend(probs.cpu().tolist())  # Collect all probs

            # Update confusion matrix
            for true, pred in zip(label.int().cpu().tolist(), (probs > 0.5).int().cpu().tolist()):
                if true == 1 and pred == 1:
                    tp += 1  # True Positive
                elif true == 0 and pred == 0:
                    tn += 1  # True Negative
                elif true == 0 and pred == 1:
                    fp += 1  # False Positive
                elif true == 1 and pred == 0:
                    fn += 1  # False Negative

    # Accuracy
    accuracy = accuracy_score(trues, preds)
    # F1 Score
    f1 = f1_score(trues, preds)
    # Specificity
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    # Sensitivity
    sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0
    # MCC
    mcc = matthews_corrcoef(trues, preds)

    # AUC - Ensure probs are in numpy format and length matches
    all_probs_np = np.array(all_probs).flatten()
    trues_np = np.array(trues)
    auc = roc_auc_score(trues_np, all_probs_np)

    print(f"Accuracy: {accuracy:.4f} | F1 Score: {f1:.4f} | Sensitivity (SN): {sensitivity:.4f} | Specificity (SP): {specificity:.4f} | MCC: {mcc:.4f} | AUC: {auc:.4f}")
    print("Confusion Matrix:\n", [[tp, fp], [fn, tn]])

    return accuracy, f1, sensitivity, specificity, mcc, auc

# === STAGE 1: ENHANCER VS NON-ENHANCER ===
def prepare_stage1():
    enhancer_train = load_fasta_clean("/kaggle/input/enhancer-detection/enhancer.cv.txt")
    non_enhancer_train = load_fasta_clean("/kaggle/input/enhancer-detection/non.cv.txt")
    
    enhancer_test = load_fasta_clean("/kaggle/input/enhancer-detection/enhancer.ind.txt")
    non_enhancer_test = load_fasta_clean("/kaggle/input/enhancer-detection/non.ind.txt")

    min_len_train = min(len(enhancer_train), len(non_enhancer_train))
    enhancer_train = enhancer_train[:min_len_train]
    non_enhancer_train = non_enhancer_train[:min_len_train]

    min_len_test = min(len(enhancer_test), len(non_enhancer_test))
    enhancer_test = enhancer_test[:min_len_test]
    non_enhancer_test = non_enhancer_test[:min_len_test]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for seq in enhancer_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(1)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(1)
    for seq in non_enhancer_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(0)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(0)

    for seq in enhancer_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(1)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(1)
    for seq in non_enhancer_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(0)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(0)

    return X_train, y_train, X_test, y_test


# === STAGE 2: STRONG VS WEAK ENHANCERS ===
def prepare_stage2():
    strong_train = load_fasta_clean("/kaggle/input/enhancer-detection/strong_742.txt")
    weak_train = load_fasta_clean("/kaggle/input/enhancer-detection/weak_742.txt")

    strong_test = load_fasta_clean("/kaggle/input/enhancer-detection/strong_100.txt")
    weak_test = load_fasta_clean("/kaggle/input/enhancer-detection/weak_100.txt")

    max_len_train = max(len(strong_train), len(weak_train))
    strong_train = (strong_train * ((max_len_train // len(strong_train)) + 1))[:max_len_train]
    weak_train = (weak_train * ((max_len_train // len(weak_train)) + 1))[:max_len_train]

    max_len_test = max(len(strong_test), len(weak_test))
    strong_test = (strong_test * ((max_len_test // len(strong_test)) + 1))[:max_len_test]
    weak_test = (weak_test * ((max_len_test // len(weak_test)) + 1))[:max_len_test]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for seq in strong_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(1)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(1)
    for seq in weak_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(0)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(0)

    for seq in strong_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(1)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(1)
    for seq in weak_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(0)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(0)

    return X_train, y_train, X_test, y_test

def train_stage1():
    tokenizer = LongformerTokenizer.from_pretrained("allenai/longformer-base-4096")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Prepare data
    X_train, y_train, X_test, y_test = prepare_stage1()
    print("Train Label Distribution:", Counter(y_train))
    print("Test Label Distribution:", Counter(y_test))

    train_ds = SimpleDataset(X_train, y_train, tokenizer)
    test_ds = SimpleDataset(X_test, y_test, tokenizer)
    batch_size = 8  # Starting batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    model = LongformerCNN(dropout_rate=0.3).to(device)  # Using Longformer + CNN
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # Learning rate scheduler (Cosine Annealing)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    best_f1 = 0
    for epoch in range(10):  # 10 epochs
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 1 - Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask).squeeze()
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")
        evaluate(model, test_loader, device)

        # Update learning rate based on scheduler
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                label = batch["label"].to(device)
                logits = model(input_ids, attention_mask).squeeze()
                preds.extend((torch.sigmoid(logits) > 0.5).int().cpu().tolist())
                trues.extend(label.int().cpu().tolist())
        f1 = f1_score(trues, preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "enhancer_stage1.pth")
            print("✅ Stage 1 model saved.")

def train_stage2():
    tokenizer = LongformerTokenizer.from_pretrained("allenai/longformer-base-4096")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Prepare data
    X_train, y_train, X_test, y_test = prepare_stage2()
    print("Train Label Distribution:", Counter(y_train))
    print("Test Label Distribution:", Counter(y_test))

    train_ds = SimpleDataset(X_train, y_train, tokenizer)
    test_ds = SimpleDataset(X_test, y_test, tokenizer)
    batch_size = 8  # Starting batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    model = LongformerCNN(dropout_rate=0.3).to(device)  # Using Longformer + CNN
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # Learning rate scheduler (Cosine Annealing)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    best_f1 = 0
    for epoch in range(10):  # 10 epochs
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 2 - Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask).squeeze()
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")
        evaluate(model, test_loader, device)

        # Update learning rate based on scheduler
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                label = batch["label"].to(device)
                logits = model(input_ids, attention_mask).squeeze()
                preds.extend((torch.sigmoid(logits) > 0.5).int().cpu().tolist())
                trues.extend(label.int().cpu().tolist())
        f1 = f1_score(trues, preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "enhancer_stage2.pth")
            print("✅ Stage 2 model saved.")

# === MAIN ===
if __name__ == "__main__":
    train_stage1()
    train_stage2()


Train Label Distribution: Counter({1: 2968, 0: 2968})
Test Label Distribution: Counter({1: 400, 0: 400})


Stage 1 - Epoch 1: 100%|██████████| 742/742 [20:36<00:00,  1.67s/it]


Epoch 1 Loss: 0.6200
Accuracy: 0.7350 | F1 Score: 0.7494 | Sensitivity (SN): 0.7925 | Specificity (SP): 0.6775 | MCC: 0.4731 | AUC: 0.7953
Confusion Matrix:
 [[317, 129], [83, 271]]
✅ Stage 1 model saved.


Stage 1 - Epoch 2: 100%|██████████| 742/742 [20:35<00:00,  1.66s/it]


Epoch 2 Loss: 0.5630
Accuracy: 0.7388 | F1 Score: 0.7461 | Sensitivity (SN): 0.7675 | Specificity (SP): 0.7100 | MCC: 0.4783 | AUC: 0.7980
Confusion Matrix:
 [[307, 116], [93, 284]]


...

Epoch 8 Loss: 0.6939
Accuracy: 0.5000 | F1 Score: 0.0000 | Sensitivity (SN): 0.0000 | Specificity (SP): 1.0000 | MCC: 0.0000 | AUC: 0.5009
Confusion Matrix:
 [[0, 0], [500, 500]]


Stage 2 - Epoch 9: 100%|██████████| 928/928 [24:28<00:00,  1.58s/it]


Epoch 9 Loss: 0.6934
Accuracy: 0.5000 | F1 Score: 0.0000 | Sensitivity (SN): 0.0000 | Specificity (SP): 1.0000 | MCC: 0.0000 | AUC: 0.6152
Confusion Matrix:
 [[0, 0], [500, 500]]


Stage 2 - Epoch 10: 100%|██████████| 928/928 [24:28<00:00,  1.58s/it]


Epoch 10 Loss: 0.6895
Accuracy: 0.5600 | F1 Score: 0.6562 | Sensitivity (SN): 0.8400 | Specificity (SP): 0.2800 | MCC: 0.1448 | AUC: 0.5565
Confusion Matrix:
 [[420, 360], [80, 140]]


LONGFORMER+BiLSTM:

In [2]:
import torch
import torch.nn as nn
import numpy as np
from transformers import LongformerTokenizer, LongformerModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, matthews_corrcoef, roc_auc_score
from collections import Counter
import random
import re
from tqdm import tqdm
import time

# --- Reproducibility ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# --- Data Utilities ---
def clean_sequence(seq):
    seq = seq.upper()
    return re.sub('[^ATCG]', '', seq)

def load_fasta_clean(file_path):
    with open(file_path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    complement = str.maketrans("ATCG", "TAGC")
    return seq.translate(complement)[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

# --- Shared Dataset Class ---
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=512):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        encoding = self.tokenizer(self.sequences[idx], padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.float),
        }

# --- Longformer + BiLSTM Model ---
class Longformer_BiLSTM(nn.Module):
    def __init__(self, dropout_rate=0.3, lstm_hidden_size=128, lstm_num_layers=2):
        super().__init__()
        # Load pre-trained Longformer model
        self.longformer = LongformerModel.from_pretrained("allenai/longformer-base-4096")
        
        # BiLSTM Layer
        self.lstm = nn.LSTM(input_size=self.longformer.config.hidden_size, 
                            hidden_size=lstm_hidden_size, 
                            num_layers=lstm_num_layers, 
                            bidirectional=True, 
                            batch_first=True)
        
        # Fully connected layer
        self.fc1 = nn.Linear(lstm_hidden_size * 2, 1)  # *2 for bidirectional LSTM
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout_rate)
        
    def forward(self, input_ids, attention_mask):
        # Pass through Longformer
        outputs = self.longformer(input_ids=input_ids, attention_mask=attention_mask)
        # Get the sequence output (not pooler output)
        x = outputs.last_hidden_state
        
        # Pass through BiLSTM
        x, _ = self.lstm(x)
        
        # Use the representation of the last hidden state
        x = x[:, -1, :]  # Get last hidden state
        
        # Pass through dropout and fully connected layer
        x = self.dropout(x)
        x = self.fc1(x)
        
        return x

# --- Evaluation Utility ---
def evaluate(model, loader, device):
    model.eval()
    preds, trues = [], []
    tp, tn, fp, fn = 0, 0, 0, 0  # Initialize confusion matrix counts
    all_probs = []  # List to store all probability values

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)
            logits = model(input_ids, attention_mask).squeeze()
            probs = torch.sigmoid(logits)
            preds.extend((probs > 0.5).int().cpu().tolist())
            trues.extend(label.int().cpu().tolist())
            all_probs.extend(probs.cpu().tolist())  # Collect all probs

            # Update confusion matrix
            for true, pred in zip(label.int().cpu().tolist(), (probs > 0.5).int().cpu().tolist()):
                if true == 1 and pred == 1:
                    tp += 1  # True Positive
                elif true == 0 and pred == 0:
                    tn += 1  # True Negative
                elif true == 0 and pred == 1:
                    fp += 1  # False Positive
                elif true == 1 and pred == 0:
                    fn += 1  # False Negative

    # Accuracy
    accuracy = accuracy_score(trues, preds)
    # F1 Score
    f1 = f1_score(trues, preds)
    # Specificity
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    # Sensitivity
    sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0
    # MCC
    mcc = matthews_corrcoef(trues, preds)

    # AUC - Ensure probs are in numpy format and length matches
    all_probs_np = np.array(all_probs).flatten()
    trues_np = np.array(trues)
    auc = roc_auc_score(trues_np, all_probs_np)

    print(f"Accuracy: {accuracy:.4f} | F1 Score: {f1:.4f} | Sensitivity (SN): {sensitivity:.4f} | Specificity (SP): {specificity:.4f} | MCC: {mcc:.4f} | AUC: {auc:.4f}")
    print("Confusion Matrix:\n", [[tp, fp], [fn, tn]])

    return accuracy, f1, sensitivity, specificity, mcc, auc

# === STAGE 1: ENHANCER VS NON-ENHANCER ===
def prepare_stage1():
    enhancer_train = load_fasta_clean("/kaggle/input/enhancer-detection/enhancer.cv.txt")
    non_enhancer_train = load_fasta_clean("/kaggle/input/enhancer-detection/non.cv.txt")
    
    enhancer_test = load_fasta_clean("/kaggle/input/enhancer-detection/enhancer.ind.txt")
    non_enhancer_test = load_fasta_clean("/kaggle/input/enhancer-detection/non.ind.txt")

    min_len_train = min(len(enhancer_train), len(non_enhancer_train))
    enhancer_train = enhancer_train[:min_len_train]
    non_enhancer_train = non_enhancer_train[:min_len_train]

    min_len_test = min(len(enhancer_test), len(non_enhancer_test))
    enhancer_test = enhancer_test[:min_len_test]
    non_enhancer_test = non_enhancer_test[:min_len_test]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for seq in enhancer_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(1)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(1)
    for seq in non_enhancer_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(0)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(0)

    for seq in enhancer_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(1)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(1)
    for seq in non_enhancer_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(0)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(0)

    return X_train, y_train, X_test, y_test


# === STAGE 2: STRONG VS WEAK ENHANCERS ===
def prepare_stage2():
    strong_train = load_fasta_clean("/kaggle/input/enhancer-detection/strong_742.txt")
    weak_train = load_fasta_clean("/kaggle/input/enhancer-detection/weak_742.txt")

    strong_test = load_fasta_clean("/kaggle/input/enhancer-detection/strong_100.txt")
    weak_test = load_fasta_clean("/kaggle/input/enhancer-detection/weak_100.txt")

    max_len_train = max(len(strong_train), len(weak_train))
    strong_train = (strong_train * ((max_len_train // len(strong_train)) + 1))[:max_len_train]
    weak_train = (weak_train * ((max_len_train // len(weak_train)) + 1))[:max_len_train]

    max_len_test = max(len(strong_test), len(weak_test))
    strong_test = (strong_test * ((max_len_test // len(strong_test)) + 1))[:max_len_test]
    weak_test = (weak_test * ((max_len_test // len(weak_test)) + 1))[:max_len_test]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for seq in strong_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(1)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(1)
    for seq in weak_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(0)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(0)

    for seq in strong_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(1)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(1)
    for seq in weak_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(0)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(0)

    return X_train, y_train, X_test, y_test

def train_stage1():
    tokenizer = LongformerTokenizer.from_pretrained("allenai/longformer-base-4096")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Prepare data
    X_train, y_train, X_test, y_test = prepare_stage1()
    print("Train Label Distribution:", Counter(y_train))
    print("Test Label Distribution:", Counter(y_test))

    train_ds = SimpleDataset(X_train, y_train, tokenizer)
    test_ds = SimpleDataset(X_test, y_test, tokenizer)
    batch_size = 8  # Starting batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    model = Longformer_BiLSTM(dropout_rate=0.3).to(device)  # Using Longformer + BiLSTM
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # Learning rate scheduler (Cosine Annealing)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    best_f1 = 0
    for epoch in range(10):  # 10 epochs
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 1 - Epoch {epoch+1}", ncols=100, dynamic_ncols=True):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask).squeeze()
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")
        evaluate(model, test_loader, device)

        # Update learning rate based on scheduler
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                label = batch["label"].to(device)
                logits = model(input_ids, attention_mask).squeeze()
                preds.extend((torch.sigmoid(logits) > 0.5).int().cpu().tolist())
                trues.extend(label.int().cpu().tolist())
        f1 = f1_score(trues, preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "enhancer_stage1.pth")
            print("✅ Stage 1 model saved.")

def train_stage2():
    tokenizer = LongformerTokenizer.from_pretrained("allenai/longformer-base-4096")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Prepare data
    X_train, y_train, X_test, y_test = prepare_stage2()
    print("Train Label Distribution:", Counter(y_train))
    print("Test Label Distribution:", Counter(y_test))

    train_ds = SimpleDataset(X_train, y_train, tokenizer)
    test_ds = SimpleDataset(X_test, y_test, tokenizer)
    batch_size = 8  # Starting batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    model = Longformer_BiLSTM(dropout_rate=0.3).to(device)  # Using Longformer + BiLSTM
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # Learning rate scheduler (Cosine Annealing)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    best_f1 = 0
    for epoch in range(10):  # 10 epochs
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 2 - Epoch {epoch+1}", ncols=100, dynamic_ncols=True):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask).squeeze()
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")
        evaluate(model, test_loader, device)

        # Update learning rate based on scheduler
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                label = batch["label"].to(device)
                logits = model(input_ids, attention_mask).squeeze()
                preds.extend((torch.sigmoid(logits) > 0.5).int().cpu().tolist())
                trues.extend(label.int().cpu().tolist())
        f1 = f1_score(trues, preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "enhancer_stage2.pth")
            print("✅ Stage 2 model saved.")

# === MAIN ===
if __name__ == "__main__":
    train_stage1()
    train_stage2()


Train Label Distribution: Counter({1: 2968, 0: 2968})
Test Label Distribution: Counter({1: 400, 0: 400})


Stage 1 - Epoch 1: 100%|██████████| 742/742 [20:39<00:00,  1.67s/it]


Epoch 1 Loss: 0.6477
Accuracy: 0.7200 | F1 Score: 0.7301 | Sensitivity (SN): 0.7575 | Specificity (SP): 0.6825 | MCC: 0.4412 | AUC: 0.7796
Confusion Matrix:
 [[303, 127], [97, 273]]
✅ Stage 1 model saved.


Stage 1 - Epoch 2: 100%|██████████| 742/742 [20:36<00:00,  1.67s/it]


Epoch 2 Loss: 0.6530
Accuracy: 0.7050 | F1 Score: 0.6731 | Sensitivity (SN): 0.6075 | Specificity (SP): 0.8025 | MCC: 0.4180 | AUC: 0.7717
Confusion Matrix:
 [[243, 79], [157, 321]]


...

Epoch 8 Loss: 0.6933
Accuracy: 0.5000 | F1 Score: 0.0000 | Sensitivity (SN): 0.0000 | Specificity (SP): 1.0000 | MCC: 0.0000 | AUC: 0.5000
Confusion Matrix:
 [[0, 0], [500, 500]]


Stage 2 - Epoch 9: 100%|██████████| 928/928 [24:30<00:00,  1.58s/it]


Epoch 9 Loss: 0.6934
Accuracy: 0.5000 | F1 Score: 0.0000 | Sensitivity (SN): 0.0000 | Specificity (SP): 1.0000 | MCC: 0.0000 | AUC: 0.5000
Confusion Matrix:
 [[0, 0], [500, 500]]


Stage 2 - Epoch 10: 100%|██████████| 928/928 [24:29<00:00,  1.58s/it]


Epoch 10 Loss: 0.6935
Accuracy: 0.5000 | F1 Score: 0.0000 | Sensitivity (SN): 0.0000 | Specificity (SP): 1.0000 | MCC: 0.0000 | AUC: 0.5000
Confusion Matrix:
 [[0, 0], [500, 500]]


LONGFORMER +transformer encoder

In [1]:
import torch
import torch.nn as nn
import numpy as np
import time
import matplotlib.pyplot as plt
from transformers import LongformerTokenizer, LongformerModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, matthews_corrcoef, recall_score, roc_curve, auc
from collections import Counter
import random, re
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm  # Add tqdm for progress bar
from torch.optim.lr_scheduler import CosineAnnealingLR  # Import the missing CosineAnnealingLR

# --- Reproducibility ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# --- Utilities ---
def clean_sequence(seq):
    return re.sub('[^ATCG]', '', seq.upper())

def load_fasta_clean(path):
    with open(path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    return seq.translate(str.maketrans("ATCG", "TAGC"))[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

def augment_sequence(seq):
    bases = ['A','T','C','G']
    seq_list = list(seq)
    for i in range(len(seq)):
        if random.random() < 0.05:
            seq_list[i] = random.choice([b for b in bases if b != seq_list[i]])
    kmers = ["".join(seq_list[i:i+6]) for i in range(len(seq_list)-5)]
    random.shuffle(kmers)
    return " ".join(kmers)

# --- Dataset ---
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=128, augment=False):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.augment = augment

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        if self.augment and random.random() < 0.5:
            seq = augment_sequence(seq)
        else:
            seq = kmer_tokenize(seq)
        encoding = self.tokenizer(seq, padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float)
        }

# --- Longformer + Transformer Encoder Model ---
class Longformer_TransformerEncoder(nn.Module):
    def __init__(self, dropout_rate=0.3, transformer_hidden_size=256, transformer_num_layers=4):
        super().__init__()
        # Load pre-trained Longformer model
        self.longformer = LongformerModel.from_pretrained("allenai/longformer-base-4096")
        
        # Transformer Encoder Layer
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=self.longformer.config.hidden_size, 
                                                        nhead=8, 
                                                        dim_feedforward=transformer_hidden_size)
        self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=transformer_num_layers)
        
        # Fully connected layer
        self.fc1 = nn.Linear(self.longformer.config.hidden_size, 1)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout_rate)
        
    def forward(self, input_ids, attention_mask):
        # Pass through Longformer
        outputs = self.longformer(input_ids=input_ids, attention_mask=attention_mask)
        # Get the sequence output (not pooler output)
        x = outputs.last_hidden_state
        
        # Pass through Transformer Encoder
        x = x.permute(1, 0, 2)  # Convert to (seq_len, batch_size, hidden_size)
        x = self.transformer_encoder(x)
        
        # Get the first token (CLS token representation) after Transformer Encoder
        x = x[0, :]  # Get the first token (CLS token representation)
        
        # Pass through dropout and fully connected layer
        x = self.dropout(x)
        x = self.fc1(x)
        
        return x

# --- Evaluation Utility ---
def evaluate(model, loader, device):
    model.eval()
    preds, trues = [], []
    tp, tn, fp, fn = 0, 0, 0, 0  # Initialize confusion matrix counts
    all_probs = []  # List to store all probability values

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)
            logits = model(input_ids, attention_mask).squeeze()
            probs = torch.sigmoid(logits)
            preds.extend((probs > 0.5).int().cpu().tolist())
            trues.extend(label.int().cpu().tolist())
            all_probs.extend(probs.cpu().tolist())  # Collect all probs

            # Update confusion matrix
            for true, pred in zip(label.int().cpu().tolist(), (probs > 0.5).int().cpu().tolist()):
                if true == 1 and pred == 1:
                    tp += 1  # True Positive
                elif true == 0 and pred == 0:
                    tn += 1  # True Negative
                elif true == 0 and pred == 1:
                    fp += 1  # False Positive
                elif true == 1 and pred == 0:
                    fn += 1  # False Negative

    # Accuracy
    accuracy = accuracy_score(trues, preds)
    # F1 Score
    f1 = f1_score(trues, preds)
    # Specificity
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    # Sensitivity
    sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0
    # MCC
    mcc = matthews_corrcoef(trues, preds)

    # AUC - Ensure probs are in numpy format and length matches
    all_probs_np = np.array(all_probs).flatten()
    trues_np = np.array(trues)
    auc = roc_auc_score(trues_np, all_probs_np)

    print(f"Accuracy: {accuracy:.4f} | F1 Score: {f1:.4f} | Sensitivity (SN): {sensitivity:.4f} | Specificity (SP): {specificity:.4f} | MCC: {mcc:.4f} | AUC: {auc:.4f}")
    print("Confusion Matrix:\n", [[tp, fp], [fn, tn]])

    return accuracy, f1, sensitivity, specificity, mcc, auc

# === STAGE 1: ENHANCER VS NON-ENHANCER ===
def prepare_stage1():
    enhancer_train = load_fasta_clean("/kaggle/input/enhancer-detection/enhancer.cv.txt")
    non_enhancer_train = load_fasta_clean("/kaggle/input/enhancer-detection/non.cv.txt")
    
    enhancer_test = load_fasta_clean("/kaggle/input/enhancer-detection/enhancer.ind.txt")
    non_enhancer_test = load_fasta_clean("/kaggle/input/enhancer-detection/non.ind.txt")

    min_len_train = min(len(enhancer_train), len(non_enhancer_train))
    enhancer_train = enhancer_train[:min_len_train]
    non_enhancer_train = non_enhancer_train[:min_len_train]

    min_len_test = min(len(enhancer_test), len(non_enhancer_test))
    enhancer_test = enhancer_test[:min_len_test]
    non_enhancer_test = non_enhancer_test[:min_len_test]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for seq in enhancer_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(1)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(1)
    for seq in non_enhancer_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(0)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(0)

    for seq in enhancer_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(1)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(1)
    for seq in non_enhancer_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(0)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(0)

    return X_train, y_train, X_test, y_test


# === STAGE 2: STRONG VS WEAK ENHANCERS ===
def prepare_stage2():
    strong_train = load_fasta_clean("/kaggle/input/enhancer-detection/strong_742.txt")
    weak_train = load_fasta_clean("/kaggle/input/enhancer-detection/weak_742.txt")

    strong_test = load_fasta_clean("/kaggle/input/enhancer-detection/strong_100.txt")
    weak_test = load_fasta_clean("/kaggle/input/enhancer-detection/weak_100.txt")

    max_len_train = max(len(strong_train), len(weak_train))
    strong_train = (strong_train * ((max_len_train // len(strong_train)) + 1))[:max_len_train]
    weak_train = (weak_train * ((max_len_train // len(weak_train)) + 1))[:max_len_train]

    max_len_test = max(len(strong_test), len(weak_test))
    strong_test = (strong_test * ((max_len_test // len(strong_test)) + 1))[:max_len_test]
    weak_test = (weak_test * ((max_len_test // len(weak_test)) + 1))[:max_len_test]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for seq in strong_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(1)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(1)
    for seq in weak_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(0)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(0)

    for seq in strong_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(1)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(1)
    for seq in weak_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(0)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(0)

    return X_train, y_train, X_test, y_test

def train_stage1():
    tokenizer = LongformerTokenizer.from_pretrained("allenai/longformer-base-4096")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Prepare data
    X_train, y_train, X_test, y_test = prepare_stage1()
    print("Train Label Distribution:", Counter(y_train))
    print("Test Label Distribution:", Counter(y_test))

    train_ds = SimpleDataset(X_train, y_train, tokenizer)
    test_ds = SimpleDataset(X_test, y_test, tokenizer)
    batch_size = 8  # Starting batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    model = Longformer_TransformerEncoder(dropout_rate=0.3).to(device)  # Using Longformer + Transformer Encoder
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # Learning rate scheduler (Cosine Annealing)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    best_f1 = 0
    for epoch in range(10):  # 15 epochs for better learning
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 1 - Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask).squeeze()
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")
        evaluate(model, test_loader, device)

        # Update learning rate based on scheduler
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                label = batch["label"].to(device)
                logits = model(input_ids, attention_mask).squeeze()
                preds.extend((torch.sigmoid(logits) > 0.5).int().cpu().tolist())
                trues.extend(label.int().cpu().tolist())
        f1 = f1_score(trues, preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "enhancer_stage1.pth")
            print("✅ Stage 1 model saved.")

def train_stage2():
    tokenizer = LongformerTokenizer.from_pretrained("allenai/longformer-base-4096")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Prepare data
    X_train, y_train, X_test, y_test = prepare_stage2()
    print("Train Label Distribution:", Counter(y_train))
    print("Test Label Distribution:", Counter(y_test))

    train_ds = SimpleDataset(X_train, y_train, tokenizer)
    test_ds = SimpleDataset(X_test, y_test, tokenizer)
    batch_size = 8  # Starting batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    model = Longformer_TransformerEncoder(dropout_rate=0.3).to(device)  # Using Longformer + Transformer Encoder
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # Learning rate scheduler (Cosine Annealing)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    best_f1 = 0
    for epoch in range(10):  # 15 epochs for better learning
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 2 - Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask).squeeze()
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")
        evaluate(model, test_loader, device)

        # Update learning rate based on scheduler
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                label = batch["label"].to(device)
                logits = model(input_ids, attention_mask).squeeze()
                preds.extend((torch.sigmoid(logits) > 0.5).int().cpu().tolist())
                trues.extend(label.int().cpu().tolist())
        f1 = f1_score(trues, preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "enhancer_stage2.pth")
            print("✅ Stage 2 model saved.")

# === MAIN ===
if __name__ == "__main__":
    train_stage1()
    train_stage2()


2025-07-19 19:21:21.990980: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752952882.195849      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752952882.256536      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

Epoch 8 Loss: 0.6960
Accuracy: 0.5440 | F1 Score: 0.6404 | Sensitivity (SN): 0.8120 | Specificity (SP): 0.2760 | MCC: 0.1042 | AUC: 0.5239
Confusion Matrix:
 [[406, 362], [94, 138]]


Stage 2 - Epoch 9: 100%|██████████| 928/928 [25:00<00:00,  1.62s/it]


Epoch 9 Loss: 0.6953
Accuracy: 0.5000 | F1 Score: 0.0000 | Sensitivity (SN): 0.0000 | Specificity (SP): 1.0000 | MCC: 0.0000 | AUC: 0.5053
Confusion Matrix:
 [[0, 0], [500, 500]]


Stage 2 - Epoch 10: 100%|██████████| 928/928 [25:00<00:00,  1.62s/it]


Epoch 10 Loss: 0.6942
Accuracy: 0.5280 | F1 Score: 0.6307 | Sensitivity (SN): 0.8060 | Specificity (SP): 0.2500 | MCC: 0.0674 | AUC: 0.4928
Confusion Matrix:
 [[403, 375], [97, 125]]


LONGFORMER+CNN+BiLSTM+ATTENTION MECHANISM

In [1]:
import torch
import torch.nn as nn
import numpy as np
from transformers import LongformerTokenizer, LongformerModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, matthews_corrcoef, roc_auc_score
from collections import Counter
import random, re
from tqdm import tqdm

# --- Reproducibility ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# --- Data Utilities ---
def clean_sequence(seq):
    seq = seq.upper()
    return re.sub('[^ATCG]', '', seq)

def load_fasta_clean(file_path):
    with open(file_path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    complement = str.maketrans("ATCG", "TAGC")
    return seq.translate(complement)[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

# --- Shared Dataset Class ---
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=512):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        encoding = self.tokenizer(self.sequences[idx], padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.float),
        }

# --- Attention Mechanism ---
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_output):
        attn_weights = torch.softmax(self.attn(lstm_output), dim=1)
        context = torch.sum(attn_weights * lstm_output, dim=1)
        return context, attn_weights

# --- Longformer + CNN + BiLSTM + Attention Mechanism ---
class Longformer_CNN_BiLSTM_Attn(nn.Module):
    def __init__(self, longformer_name="allenai/longformer-base-4096", hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.longformer = LongformerModel.from_pretrained(longformer_name)
        
        # CNN Layer
        self.cnn = nn.Sequential(
            nn.Conv1d(self.longformer.config.hidden_size, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)
        )
        
        # BiLSTM Layer
        self.bilstm = nn.LSTM(64, hidden_dim, num_layers=num_layers, batch_first=True, bidirectional=True)
        
        # Attention Mechanism
        self.attn = Attention(hidden_dim * 2)  # Because BiLSTM is bidirectional
        
        # Final classification layer
        self.classifier = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask):
        # Get the hidden states from Longformer
        outputs = self.longformer(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state  # [batch_size, seq_len, hidden_size]

        # Apply CNN
        cnn_input = hidden_states.permute(0, 2, 1)  # Change to [batch_size, hidden_size, seq_len]
        cnn_out = self.cnn(cnn_input)  # [batch_size, num_filters, seq_len]
        cnn_out = cnn_out.permute(0, 2, 1)  # [batch_size, seq_len, num_filters]
        
        # Apply BiLSTM
        lstm_out, _ = self.bilstm(cnn_out)  # [batch_size, seq_len, hidden_dim * 2]

        # Apply Attention Mechanism
        context, _ = self.attn(lstm_out)  # [batch_size, hidden_dim * 2]

        # Pass through dropout and final classifier
        context = self.dropout(context)
        logits = self.classifier(context).squeeze(-1)
        return logits

# --- Evaluation Utility ---
def evaluate(model, loader, device):
    model.eval()
    preds, trues = [], []
    tp, tn, fp, fn = 0, 0, 0, 0  # Initialize confusion matrix counts
    all_probs = []  # List to store all probability values

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)
            logits = model(input_ids, attention_mask).squeeze()
            probs = torch.sigmoid(logits)
            preds.extend((probs > 0.5).int().cpu().tolist())
            trues.extend(label.int().cpu().tolist())
            all_probs.extend(probs.cpu().tolist())  # Collect all probs

            # Update confusion matrix
            for true, pred in zip(label.int().cpu().tolist(), (probs > 0.5).int().cpu().tolist()):
                if true == 1 and pred == 1:
                    tp += 1  # True Positive
                elif true == 0 and pred == 0:
                    tn += 1  # True Negative
                elif true == 0 and pred == 1:
                    fp += 1  # False Positive
                elif true == 1 and pred == 0:
                    fn += 1  # False Negative

    # Accuracy
    accuracy = accuracy_score(trues, preds)
    # F1 Score
    f1 = f1_score(trues, preds)
    # Specificity
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    # Sensitivity
    sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0
    # MCC
    mcc = matthews_corrcoef(trues, preds)

    # AUC - Ensure probs are in numpy format and length matches
    all_probs_np = np.array(all_probs).flatten()
    trues_np = np.array(trues)
    auc = roc_auc_score(trues_np, all_probs_np)

    print(f"Accuracy: {accuracy:.4f} | F1 Score: {f1:.4f} | Sensitivity (SN): {sensitivity:.4f} | Specificity (SP): {specificity:.4f} | MCC: {mcc:.4f} | AUC: {auc:.4f}")
    print("Confusion Matrix:\n", [[tp, fp], [fn, tn]])

    return accuracy, f1, sensitivity, specificity, mcc, auc

# === STAGE 1: ENHANCER VS NON-ENHANCER ===
def prepare_stage1():
    enhancer_train = load_fasta_clean("/kaggle/input/enhancer-detection/enhancer.cv.txt")
    non_enhancer_train = load_fasta_clean("/kaggle/input/enhancer-detection/non.cv.txt")
    
    enhancer_test = load_fasta_clean("/kaggle/input/enhancer-detection/enhancer.ind.txt")
    non_enhancer_test = load_fasta_clean("/kaggle/input/enhancer-detection/non.ind.txt")

    min_len_train = min(len(enhancer_train), len(non_enhancer_train))
    enhancer_train = enhancer_train[:min_len_train]
    non_enhancer_train = non_enhancer_train[:min_len_train]

    min_len_test = min(len(enhancer_test), len(non_enhancer_test))
    enhancer_test = enhancer_test[:min_len_test]
    non_enhancer_test = non_enhancer_test[:min_len_test]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for seq in enhancer_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(1)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(1)
    for seq in non_enhancer_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(0)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(0)

    for seq in enhancer_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(1)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(1)
    for seq in non_enhancer_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(0)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(0)

    return X_train, y_train, X_test, y_test


# === STAGE 2: STRONG VS WEAK ENHANCERS ===
def prepare_stage2():
    strong_train = load_fasta_clean("/kaggle/input/enhancer-detection/strong_742.txt")
    weak_train = load_fasta_clean("/kaggle/input/enhancer-detection/weak_742.txt")

    strong_test = load_fasta_clean("/kaggle/input/enhancer-detection/strong_100.txt")
    weak_test = load_fasta_clean("/kaggle/input/enhancer-detection/weak_100.txt")

    max_len_train = max(len(strong_train), len(weak_train))
    strong_train = (strong_train * ((max_len_train // len(strong_train)) + 1))[:max_len_train]
    weak_train = (weak_train * ((max_len_train // len(weak_train)) + 1))[:max_len_train]

    max_len_test = max(len(strong_test), len(weak_test))
    strong_test = (strong_test * ((max_len_test // len(strong_test)) + 1))[:max_len_test]
    weak_test = (weak_test * ((max_len_test // len(weak_test)) + 1))[:max_len_test]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for seq in strong_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(1)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(1)
    for seq in weak_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(0)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(0)

    for seq in strong_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(1)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(1)
    for seq in weak_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(0)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(0)

    return X_train, y_train, X_test, y_test

def train_stage1():
    tokenizer = LongformerTokenizer.from_pretrained("allenai/longformer-base-4096")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Prepare data
    X_train, y_train, X_test, y_test = prepare_stage1()
    print("Train Label Distribution:", Counter(y_train))
    print("Test Label Distribution:", Counter(y_test))

    train_ds = SimpleDataset(X_train, y_train, tokenizer)
    test_ds = SimpleDataset(X_test, y_test, tokenizer)
    batch_size = 8  # Starting batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    model = Longformer_CNN_BiLSTM_Attn().to(device)  # Using Longformer + CNN + BiLSTM + Attention
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=1e-5)

    # Learning rate scheduler (Cosine Annealing)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    best_f1 = 0
    for epoch in range(10):  # 10 epochs in Stage 1
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 1 - Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask).squeeze()
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")
        evaluate(model, test_loader, device)

        # Update learning rate based on scheduler
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                label = batch["label"].to(device)
                logits = model(input_ids, attention_mask).squeeze()
                preds.extend((torch.sigmoid(logits) > 0.5).int().cpu().tolist())
                trues.extend(label.int().cpu().tolist())
        f1 = f1_score(trues, preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "enhancer_stage1.pth")
            print("✅ Stage 1 model saved.")

def train_stage2():
    tokenizer = LongformerTokenizer.from_pretrained("allenai/longformer-base-4096")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Prepare data
    X_train, y_train, X_test, y_test = prepare_stage2()
    print("Train Label Distribution:", Counter(y_train))
    print("Test Label Distribution:", Counter(y_test))

    train_ds = SimpleDataset(X_train, y_train, tokenizer)
    test_ds = SimpleDataset(X_test, y_test, tokenizer)
    batch_size = 8  # Starting batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    model = Longformer_CNN_BiLSTM_Attn().to(device)  # Using Longformer + CNN + BiLSTM + Attention
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=1e-5)

    # Learning rate scheduler (Cosine Annealing)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    best_f1 = 0
    for epoch in range(10):  # 10 epochs in Stage 2
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 2 - Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask).squeeze()
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")
        evaluate(model, test_loader, device)

        # Update learning rate based on scheduler
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                label = batch["label"].to(device)
                logits = model(input_ids, attention_mask).squeeze()
                preds.extend((torch.sigmoid(logits) > 0.5).int().cpu().tolist())
                trues.extend(label.int().cpu().tolist())
        f1 = f1_score(trues, preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "enhancer_stage2.pth")
            print("✅ Stage 2 model saved.")

# === MAIN ===
if __name__ == "__main__":
    train_stage1()
    train_stage2()


2025-07-21 04:50:59.634249: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753073459.838851      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753073459.902192      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

Epoch 8 Loss: 0.6917
Accuracy: 0.5320 | F1 Score: 0.5880 | Sensitivity (SN): 0.6680 | Specificity (SP): 0.3960 | MCC: 0.0665 | AUC: 0.6008
Confusion Matrix:
 [[334, 302], [166, 198]]


Stage 2 - Epoch 9: 100%|██████████| 928/928 [24:18<00:00,  1.57s/it]


Epoch 9 Loss: 0.6891
Accuracy: 0.4400 | F1 Score: 0.5096 | Sensitivity (SN): 0.5820 | Specificity (SP): 0.2980 | MCC: -0.1252 | AUC: 0.5234
Confusion Matrix:
 [[291, 351], [209, 149]]


Stage 2 - Epoch 10: 100%|██████████| 928/928 [24:19<00:00,  1.57s/it]


Epoch 10 Loss: 0.6882
Accuracy: 0.4600 | F1 Score: 0.4000 | Sensitivity (SN): 0.3600 | Specificity (SP): 0.5600 | MCC: -0.0816 | AUC: 0.5361
Confusion Matrix:
 [[180, 220], [320, 280]]
